# Process the Perturb-seq datasets

Runs PIE's pipeline on the eight cell-level datasets, so that every model is trained and scored on the same labels:

1. Keep the cells of each dataset's knockdowns in the development folds (listed in [`splits/`](splits/)), and every non-targeting control cell. Fold 4 stays out until the end.
2. Choose the readout genes: TxPert's 5,000 genes for Replogle K562, and the 5,000 most variable genes (seurat_v3, at most 200,000 cells) for the others. PIE predicts the readouts it has gene text for.
3. Scale each cell to 10,000 counts and take log(1 + x) (`pie process`).
4. Call the DE genes of each knockdown against the non-targeting cells of its context, cell by cell (`pie process`, FDR 0.05).
5. Write each knockdown's expression change, DE calls and fold changes on the readout genes to `perturbseq/processed/<dataset>/preprocessed/` (`pie prep`).

Steps 3 to 5 run PIE's `pie` command and need one GPU. The notebook runs in the `pwpa` kernel and takes `pie` from the `PIE_BIN` environment variable (the `pie` executable of PIE's own environment), else from `PATH`.
Set `DATASETS` to a comma-separated list to process only some datasets, for example one per Slurm array task; by default it processes all of them and Marson.
Genes are named the way PIE looks them up: by HGNC symbol for human genes and by the human symbol of the one-to-one orthologue for mouse genes.
The HGNC and MGI tables for this are downloaded into `perturbseq/raw/reference/`.

Marson T cells also get pseudobulk responses, for [`3.further_analysis`](../3.further_analysis/).

In [ ]:
import hashlib
import json
import os
import zipfile
import shutil
import subprocess
import sys
import urllib.request

import anndata as ad
import h5py
import numpy as np
import pandas as pd
import scanpy as sc
from anndata.io import read_elem, sparse_dataset
from scipy import sparse

sys.path.append("../utils")
from paths import data_dir

## Shared steps

In [ ]:
RAW, REF = data_dir("perturbseq", "raw"), data_dir("perturbseq", "raw", "reference")
CTRL = "non-targeting"
PIE = os.environ.get("PIE_BIN") or shutil.which("pie")
datasets = ["replogle_k562", "replogle_rpe1", "nadig_hepg2", "nadig_jurkat", "kolf_ipsc", "vcc_h1", "shi_brain", "geiger_bmdc", "marson"]
datasets = [d for d in datasets if d in os.environ.get("DATASETS", ",".join(datasets)).split(",")]

# the gene naming tables; the hashes are of the copies behind the published results, and both URLs serve the current release
for url, sha in [("https://storage.googleapis.com/public-download-files/hgnc/tsv/tsv/hgnc_complete_set.txt",
                  "2b4224ea847df2fc6982f5b2a52804c5d92fbb8810b134afb636f6029452dc03"),
                 ("https://www.informatics.jax.org/downloads/reports/HOM_MouseHumanSequence.rpt",
                  "d57ab51934659e3edcd0fd51248b71061a3024c0838affb1ea43842ada72970f")]:
    path = REF / url.rsplit("/", 1)[1]
    if not path.exists():
        urllib.request.urlretrieve(url, path)
    if hashlib.sha256(path.read_bytes()).hexdigest() != sha:
        print(f"{path.name} is a newer release than the published results used: a few gene names may differ")

KEYS = set(json.loads((data_dir("priors", "raw", "gene_text") / "meta.json").read_text())["keys"])  # genes PIE has text for
HGNC = pd.read_csv(REF / "hgnc_complete_set.txt", sep="\t", dtype=str, usecols=["symbol", "ensembl_gene_id"]).dropna()
HGNC = HGNC[~HGNC.ensembl_gene_id.duplicated(keep=False)]  # one-to-one ENSG -> symbol only
HOM = pd.read_csv(REF / "HOM_MouseHumanSequence.rpt", sep="\t", dtype=str)


def splits(dataset):
    return pd.read_csv(f"splits/{dataset}.tsv", sep="\t")


def obs_var(path):
    with h5py.File(path, "r") as f:
        return read_elem(f["obs"]), read_elem(f["var"])


def counts(path, rows, key="X"):
    """Raw counts of the sorted rows `rows` of an h5ad's dense, CSR or CSC matrix, as float32 CSR."""
    with h5py.File(path, "r") as f:
        X = f[key]
        if isinstance(X, h5py.Dataset):  # dense: read in windows of rows
            w = 10_000
            M = sparse.vstack([sparse.csr_matrix(X[s:s + w][rows[(rows >= s) & (rows < s + w)] - s]) for s in range(0, X.shape[0], w)])
        elif X.attrs["encoding-type"] == "csc_matrix":  # read in windows of genes
            X = sparse_dataset(X)
            M = sparse.hstack([X[:, s:s + 2000].tocsr()[rows] for s in range(0, X.shape[1], 2000)])
        else:
            M = sparse_dataset(X)[rows]
    M = sparse.csr_matrix(M, dtype=np.float32)
    assert (M.data >= 0).all() and np.array_equal(M.data, np.round(M.data)), f"{path} {key} is not raw counts"
    return M


def counts_files(paths, cells):
    """Raw counts of the cells (columns file, row) of several files, in the cells' order (sorted by file, then row)."""
    return sparse.vstack([counts(p, cells.row.values[cells.file.values == i]) for i, p in enumerate(paths)]).tocsr()


def sample(rows, n=200_000):
    """At most n of the rows: the seed-0 draw the readout genes are chosen on."""
    return np.sort(np.random.default_rng(0).choice(rows, n, replace=False)) if len(rows) > n else rows


def hvg(X):
    """The 5,000 most variable genes on raw counts (seurat_v3), as a mask in var order."""
    a = ad.AnnData(X)
    sc.pp.highly_variable_genes(a, flavor="seurat_v3", n_top_genes=5000)
    return a.var.highly_variable.values


def human_names(ensg, symbol):
    """The HGNC symbol of the gene's ENSG if PIE has text for it, else its raw symbol if PIE has text for that.
    A name two genes would share, or no name, becomes x_<ENSG>."""
    ensg, symbol = pd.Series(ensg), pd.Series(symbol)
    h = ensg.map(dict(zip(HGNC.ensembl_gene_id, HGNC.symbol)))
    name = h.where(h.isin(KEYS), symbol.where(symbol.isin(KEYS)))
    return np.where(name.notna() & ~name.duplicated(keep=False), name, "x_" + ensg).astype(str)


def mouse_names(symbol):
    """The human symbol of the gene's one-to-one MGI orthologue (matched on the mouse symbol), else mm_<symbol>."""
    mouse, human = HOM["Common Organism Name"].str.startswith("mouse"), HOM["Common Organism Name"] == "human"
    k = HOM["DB Class Key"]
    one = k.map(k[mouse].value_counts()).eq(1) & k.map(k[human].value_counts()).eq(1)
    h = pd.Series(symbol).map(dict(zip(HOM.Symbol[one & mouse], k[one & mouse]))).map(dict(zip(k[one & human], HOM.Symbol[one & human])))
    return np.where(h.notna() & ~h.duplicated(keep=False), h, "mm_" + pd.Series(symbol)).astype(str)


def pie_cells(cells, dataset, col):
    """The cells of the dataset's development rows (context, knockdown) and every control cell, with PIE's obs.
    `cells` has columns context, label (matched to the splits' column `col`) and ctrl."""
    s = splits(dataset)
    dev = s[s.fold < 4].set_index(["context", col], drop=False)  # keeps symbol and ensg as columns for the obs below
    keep = pd.MultiIndex.from_arrays([cells.context, cells.label]).isin(dev.index) | cells.ctrl.values
    c = cells[keep]
    r = dev.reindex(pd.MultiIndex.from_arrays([c.context, c.label]))
    assert set(dev.index) == set(zip(c.context[~c.ctrl], c.label[~c.ctrl])), "a development row without cells"
    obs = pd.DataFrame({"context": c.context.values, "target_gene": np.where(c.ctrl, CTRL, r.symbol.values),
                        "gene_id": np.where(c.ctrl, "", r.ensg.values)}, index=np.arange(len(c)).astype(str))
    return c, obs


def to_pie(dataset, obs, X, ensg, names, readout):
    """Write PIE's input (raw counts of every kept cell) and gene axis, then run `pie process` and `pie prep`."""
    assert PIE, "set PIE_BIN to the pie executable of PIE's environment"
    out = data_dir("perturbseq", "processed", dataset)
    assert pd.Index(names).is_unique and np.asarray(X.sum(1)).min() > 0, "a cell with no counts"
    a = ad.AnnData(X, obs=obs, var=pd.DataFrame({"ensg": ensg}, index=names))
    a.write_h5ad(data_dir("perturbseq", "processed", dataset, "counts") / f"{dataset}.h5ad")
    axis = pd.DataFrame({"axis": names, "readout": ensg})[readout]
    axis = axis[axis.axis.isin(KEYS)]
    (out / "axis.txt").write_text("\n".join(axis.axis) + "\n")
    print(dataset, a.shape, obs.groupby(["context", obs.target_gene == CTRL]).size().to_dict(), "axis", len(axis), flush=True)

    env = {**os.environ, "PIE_DATA_ROOT": f"{out}/pie_data", "PIE_RUNS_ROOT": f"{out}/runs", "PIE_CACHE_DIR": f"{out}/pie_cache"}
    pie = lambda *args: subprocess.run([PIE, *args], env=env, check=True)  # noqa: E731
    pie("process", "overwrite=true", f"input={out}/counts/{dataset}.h5ad", "normalize.enabled=true",
        f"normalize.output_dir={out}/expression", "de.enabled=true", "de.groupby=target_gene", f"de.reference={CTRL}",
        "de.context_column=context", f"de.output_dir={out}/de")
    pie("prep", "overwrite=true", f"name={dataset}", "label_format=pie_process", f"labels={out}/de/*.parquet",
        f"h5ad={out}/expression/*.h5ad", f"output_dir={out}/preprocessed", f"genes={out}/axis.txt",
        "obs.context_col=context", "obs.pert_col=target_gene", "obs.pert_id_col=gene_id", f"obs.control_label={CTRL}")
    axis.to_parquet(out / "preprocessed/axis_map.parquet", index=False)  # each readout's gene id, in meta.json's order

## The cells of each dataset

Each dataset keeps the authors' cell QC and guide assignment, with the few extra rules noted below.
The readout genes are chosen on the cells of every kept knockdown, all five folds, and the controls, unless noted.

In [ ]:
def replogle_nadig(dataset, file):
    """Replogle and Nadig: one file, dense raw counts, the cell's knockdown in obs `gene`."""
    path = RAW / dataset / file
    a = ad.read_h5ad(path, backed="r")  # an old anndata layout
    gene = a.obs.gene.astype(str).values
    ensg, symbol = a.var.index.str.split(".").str[0].values, a.var.gene_name.astype(str).values
    a.file.close()
    s = splits(dataset)
    cells = pd.DataFrame({"row": np.arange(len(gene)), "context": s.context[0], "label": gene, "ctrl": gene == CTRL})
    c, obs = pie_cells(cells, dataset, "symbol")
    if dataset == "replogle_k562":  # TxPert's 5,000 genes; TxPert names the second copy of a repeated symbol "<symbol>-1"
        tx_file = RAW / dataset / "K562_single_cell_line/de_adata_test.h5ad"
        if not tx_file.exists():  # from TxPert's K562_single_cell_line.zip (0.download_perturbseq.ipynb)
            with zipfile.ZipFile(RAW / dataset / "K562_single_cell_line.zip") as z:
                member = next(m for m in z.namelist() if m.endswith("de_adata_test.h5ad"))
                tx_file.parent.mkdir(exist_ok=True)
                tx_file.write_bytes(z.read(member))
        tx = ad.read_h5ad(tx_file, backed="r").var_names
        readout = ad.utils.make_index_unique(pd.Index(symbol)).isin(tx)
    else:
        readout = hvg(counts(path, sample(np.flatnonzero(np.isin(gene, s.symbol) | cells.ctrl))))
    return obs, counts(path, c.row.values), ensg, human_names(ensg, symbol), readout


def kolf_ipsc(dataset):
    """KOLF: one file; X is z-scored, so the raw counts are layers/counts. Controls are pooled over the 3 batches.
    Readouts are chosen on 200,000 of all released cells."""
    path = RAW / dataset / "KOLF_Strong_Perturbations.h5ad"
    o, v = obs_var(path)
    cells = pd.DataFrame({"row": np.arange(len(o)), "context": splits(dataset).context[0],
                          "label": o.gene_target_ensembl_id.astype(str).values, "ctrl": (o.gene_target == "NTC").values})
    c, obs = pie_cells(cells, dataset, "ensg")
    readout = hvg(counts(path, sample(np.arange(len(o))), "layers/counts"))
    return obs, counts(path, c.row.values, "layers/counts"), v.gene_ids.values, human_names(v.gene_ids.values, v.index.values), readout


def vcc_h1(dataset):
    """VCC: three files that repeat the same control cells (kept once); targets without an ENSG are dropped.
    A knockdown's cells are those of its guide x lane-block samples with at least 5 cells, as for the pseudobulks.
    Readouts are chosen on 200,000 of these cells before the 5-cell rule."""
    paths = [RAW / dataset / f"adata_{s}.h5ad" for s in ("Training", "Validation", "Test")]
    ov = [obs_var(p) for p in paths]
    var = ov[0][1]
    s2e = dict(zip(var.index, var.gene_id))
    cells = pd.concat([pd.DataFrame({"file": i, "row": np.arange(len(o)), "barcode": o.index.astype(str), "lane": o.batch.astype(str).values,
                                     "guide": o.guide_id.astype(str).values, "target": o.target_gene.astype(str).values})
                       for i, (o, _) in enumerate(ov)], ignore_index=True)
    lane = cells.lane.str.extract(r"^Flex_(\d+)_(\d+)$").astype(int)
    first = lane.groupby(0)[1].rank(method="dense") <= (lane.groupby(0)[1].transform("nunique") + 1) // 2
    cells["block"] = lane[0].astype(str) + np.where(first, "a", "b")  # each run's lanes, sorted, in two halves
    cells = cells[~cells.barcode.duplicated() & (cells.target.eq(CTRL) | cells.target.isin(s2e))]
    cells = cells.assign(context=splits(dataset).context[0], label=cells.target.map(s2e), ctrl=cells.target.eq(CTRL))
    readout = hvg(counts_files(paths, cells.iloc[sample(np.arange(len(cells)))]))
    c, obs = pie_cells(cells[cells.groupby(["guide", "block"]).guide.transform("size") >= 5], dataset, "ensg")
    return obs, counts_files(paths, c), var.gene_id.values, human_names(var.gene_id.values, var.index.values), readout


def shi_brain(dataset):
    """Shi: 14 channel files; cells that pass QC with one guide, in the 3 Allen classes with a median of at least 100
    cells per target gene. Safe-targeting cells are dropped, and the guide-capture features too.
    Readouts are chosen on 200,000 of these cells."""
    files = ["1_1", "1_2", "1_3", "2_1", "2_2", "2_3", "3_1", "3_2", "3_3", "3_4", "4_1", "4_2", "4_3", "4_4"]
    paths = [RAW / dataset / f"WB8588_{fl}.h5ad" for fl in files]
    states = {"01 IT-ET Glut": "IT_ET_Glut", "18 TH Glut": "TH_Glut", "07 CTX-MGE GABA": "CTX_MGE_GABA"}
    parts = []
    for i, p in enumerate(paths):
        o, v = obs_var(p)
        label, cls = o.gene_target.astype(str), o.predicted_class.astype(str)
        k = (o.passes_qc.astype(bool) & (o.num_guides == 1) & cls.isin(states) & ~label.str.startswith("Safe_target")).values
        parts.append(pd.DataFrame({"file": i, "row": np.flatnonzero(k), "context": cls[k].map(states).values, "label": label[k].values}))
    cells = pd.concat(parts, ignore_index=True)
    cells["ctrl"] = cells.label == "Non_target"
    ge = (v.feature_types == "Gene Expression").values
    readout = hvg(counts_files(paths, cells.iloc[sample(np.arange(len(cells)))])[:, ge])
    c, obs = pie_cells(cells, dataset, "mouse_symbol")
    ensg = v.gene_ids.str.split(".").str[0].values[ge]
    return obs, counts_files(paths, c)[:, ge], ensg, mouse_names(v.index.values[ge]), readout


def geiger_bmdc(dataset):
    """Geiger: one file with a UMI column per guide in obs (released counts start at 2 UMIs, so non-zero = detected).
    A cell is kept when its detected guides all fall in one class: one target gene, or NO_TARGET (the controls)."""
    path = RAW / dataset / "GSE327057_BMDC_E3_screen_raw.h5ad"
    with h5py.File(path, "r") as f:
        guides = np.asarray(read_elem(f["uns/feature_barcode_names"])).astype(str)
        cls = pd.Series(guides).str.rsplit("_", n=1).str[0].values
        classes = np.unique(cls)
        hit = np.zeros((f["obs"][f["obs"].attrs["_index"]].shape[0], len(classes)), bool)
        for g, j in zip(guides, np.searchsorted(classes, cls)):
            hit[:, j] |= np.asarray(read_elem(f["obs"][g])) > 0
        v = read_elem(f["var"])
    one = hit.sum(1) == 1
    label = classes[hit[one].argmax(1)]
    s = splits(dataset)
    cells = pd.DataFrame({"row": np.flatnonzero(one), "context": s.context[0], "label": label, "ctrl": label == "NO_TARGET"})
    c, obs = pie_cells(cells, dataset, "mouse_symbol")
    readout = hvg(counts(path, sample(cells.row.values[cells.label.isin(s.mouse_symbol) | cells.ctrl]), "layers/counts"))
    return obs, counts(path, c.row.values, "layers/counts"), v.gene_ids.values, mouse_names(v.index.values), readout

In [ ]:
readers = {
    "replogle_k562": lambda d: replogle_nadig(d, "K562_essential_raw_singlecell_01.h5ad"),
    "replogle_rpe1": lambda d: replogle_nadig(d, "rpe1_raw_singlecell_01.h5ad"),
    "nadig_hepg2": lambda d: replogle_nadig(d, "GSE264667_hepg2_raw_singlecell_01.h5ad"),
    "nadig_jurkat": lambda d: replogle_nadig(d, "GSE264667_jurkat_raw_singlecell_01.h5ad"),
    "kolf_ipsc": kolf_ipsc,
    "vcc_h1": vcc_h1,
    "shi_brain": shi_brain,
    "geiger_bmdc": geiger_bmdc,
}
for dataset in datasets:
    if dataset in readers:
        to_pie(dataset, *readers[dataset](dataset))

## Marson pseudobulk responses

The knockdowns are those that pass the paper's filter (more than 75 DE genes and more than 50 cells), in any of the three states.
The samples are the authors' pseudobulks (one per guide, donor and state) that pass their `keep_for_DE` flag.
Each sample is scaled to 10,000 counts over the readout genes and log-transformed (`pseudobulk.h5ad`).
A knockdown's response is the unweighted mean over its samples minus the mean over the state's non-targeting samples, with its own gene's column set to 0 (`delta.npy`, in the order of `obs.parquet`).
The readout genes are those in both of the authors' DE tables (`genes.npy`), and `symbols.json` names every gene the screen targeted.

In [ ]:
def marson():
    src, out = data_dir("perturbseq", "raw", "marson"), data_dir("perturbseq", "processed", "marson")
    de = pd.read_csv(src / "DE_stats.suppl_table.csv")
    symbols = de.drop_duplicates("target_contrast").set_index("target_contrast").target_contrast_gene_name.to_dict()  # every screened gene
    de = de[(de.n_total_de_genes > 75) & (de.n_cells_target > 50)]
    obs = pd.DataFrame({"key": de["index"].values, "ensg": de.target_contrast.values, "symbol": de.target_contrast_gene_name.values,
                        "condition": de.culture_condition.values})

    with h5py.File(src / "GWCD4i.DE_stats.h5ad", "r") as f, h5py.File(src / "GWCD4i.DE_stats.by_donors.h5mu", "r") as fd:
        mvar = read_elem(f["var"]).index  # only the gene lists of the two DE tables are read
        dvar = read_elem(fd["mod"][sorted(fd["mod"])[0]]["var"]).index
    genes = mvar[mvar.isin(dvar)]
    obs["target_col"] = obs.ensg.map({g: i for i, g in enumerate(genes)}).fillna(-1).astype(int)

    with h5py.File(src / "GWCD4i.pseudobulk_merged.h5ad", "r") as f:
        pb, var = read_elem(f["obs"]), read_elem(f["var"])
        ctrl = (pb.guide_type.astype(str) == "non-targeting").values
        key = (pb.perturbed_gene_id.astype(str) + "_" + pb.culture_condition.astype(str)).values
        rows = np.flatnonzero(pb.keep_for_DE.values & (ctrl | np.isin(key, obs.key)))
        cols = var.index.get_indexer(genes)
        assert (cols >= 0).all(), "a readout gene missing from the pseudobulk counts"  # -1 would silently read the last gene
        C = sparse.csr_matrix(sparse_dataset(f["X"])[rows])[:, cols].astype(np.float32)
    X = np.log1p((sparse.diags(1e4 / np.asarray(C.sum(1)).ravel()) @ C).toarray().astype(np.float32))
    pb, ctrl, key = pb.iloc[rows], ctrl[rows], key[rows]
    cond = pb.culture_condition.astype(str).values

    kd = np.flatnonzero(~ctrl)
    keys, inv = np.unique(key[kd], return_inverse=True)
    mean = sparse.csr_matrix((1 / np.bincount(inv)[inv], (inv, kd)), shape=(len(keys), len(X))) @ X  # unweighted over samples
    ctrl_mean = {c: X[(cond == c) & ctrl].mean(0) for c in obs.condition.unique()}
    delta = mean[np.searchsorted(keys, obs.key)] - np.stack([ctrl_mean[c] for c in obs.condition])
    t = obs.target_col.values
    delta[np.flatnonzero(t >= 0), t[t >= 0]] = 0
    assert np.isin(obs.key, keys).all() and np.isfinite(delta).all()
    print(obs.condition.value_counts().to_dict(), "samples", len(X), "of them controls", int(ctrl.sum()), "readouts", len(genes))

    obs.to_parquet(out / "obs.parquet", index=False)
    np.save(out / "delta.npy", delta.astype(np.float32))
    np.save(out / "genes.npy", np.asarray(genes).astype(str))
    (out / "symbols.json").write_text(json.dumps(symbols))
    ad.AnnData(X, obs=pd.DataFrame({"ensg": np.where(ctrl, "ctrl", pb.perturbed_gene_id.astype(str)), "condition": cond,
                                     "donor": pb.donor_id.astype(str).values}, index=pb.index.astype(str)),
               var=pd.DataFrame(index=np.asarray(genes).astype(str))).write_h5ad(out / "pseudobulk.h5ad")


if "marson" in datasets:
    marson()